In [1]:
from langgraph.graph import StateGraph, START, END, MessagesState
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langgraph.checkpoint.postgres import PostgresSaver
import os
from psycopg import Connection
from langchain_core.messages.utils import trim_messages,count_tokens_approximately
from langchain.messages import RemoveMessage

In [2]:
load_dotenv()

True

In [3]:
llm = ChatGroq(model= "llama-3.1-8b-instant")

In [5]:
def chat(state: MessagesState):
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

def delete_old_messages(state: MessagesState):
    messages = state['messages']

    if len(messages) > 10:
        to_remove = messages[:-4]
        return {'messages': [RemoveMessage(id= m.id) for m in to_remove]}
    
    return {}

In [6]:
builder = StateGraph(MessagesState)

builder.add_node("chat", chat)
builder.add_node("cleanup", delete_old_messages)

builder.add_edge(START, "chat")
builder.add_edge('chat', "cleanup")
builder.add_edge('cleanup', END)


In [7]:
DB_URI = os.getenv("DATABASE_URL")

In [8]:
conn = Connection.connect(
    DB_URI,
    autocommit= True,
    prepare_threshold= None 
)

In [9]:
checkpointer = PostgresSaver(conn)
checkpointer.setup()

In [10]:
graph = builder.compile(checkpointer=checkpointer)

# Thread 1 (remembers)
t1 = {"configurable": {"thread_id": "thread-1"}}
graph.invoke({"messages": [{"role": "user", "content": "Hi, my name is Aryan"}]}, t1)
out1 = graph.invoke({"messages": [{"role": "user", "content": "What is my name?"}]}, t1)
print("Thread-1:", out1["messages"][-1].content)

Thread-1: Your name is Aryan.


In [15]:
graph.get_state(t1).values

{'messages': [HumanMessage(content='Hi, my name is Aryan', additional_kwargs={}, response_metadata={}, id='3d1b463f-3249-411a-9ffb-6e65a76f72ff'),
  AIMessage(content="Hi Aryan, we've had this conversation before. It's nice to chat with you again. Is there something specific you'd like to talk about or would you like to just chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 162, 'total_tokens': 201, 'completion_time': 0.086200738, 'completion_tokens_details': None, 'prompt_time': 0.013695901, 'prompt_tokens_details': None, 'queue_time': 0.050532683, 'total_time': 0.099896639}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e64e2-5623-7a71-9928-0a90a31b1e58-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 162, 'output_tokens': 39, 'total_tokens': 201}),
  HumanMess

In [16]:
graph.invoke({"messages": [{"role": "user", "content": "I am a SWE"}]}, t1)
graph.get_state(t1).values
graph.invoke({"messages": [{"role": "user", "content": "I live in Noida"}]}, t1)
graph.get_state(t1).values
graph.invoke({"messages": [{"role": "user", "content": "I live in Noida"}]}, t1)
graph.get_state(t1).values

{'messages': [HumanMessage(content='Hi, my name is Aryan', additional_kwargs={}, response_metadata={}, id='3d1b463f-3249-411a-9ffb-6e65a76f72ff'),
  AIMessage(content="Hi Aryan, we've had this conversation before. It's nice to chat with you again. Is there something specific you'd like to talk about or would you like to just chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 162, 'total_tokens': 201, 'completion_time': 0.086200738, 'completion_tokens_details': None, 'prompt_time': 0.013695901, 'prompt_tokens_details': None, 'queue_time': 0.050532683, 'total_time': 0.099896639}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e64e2-5623-7a71-9928-0a90a31b1e58-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 162, 'output_tokens': 39, 'total_tokens': 201}),
  HumanMess

In [17]:
graph.invoke({"messages": [{"role": "user", "content": "I am doing B. Tech"}]}, t1)
graph.get_state(t1).values

{'messages': [HumanMessage(content='I live in Noida', additional_kwargs={}, response_metadata={}, id='efc7ee25-f4d7-4a46-9bed-331b21cf7a92'),
  AIMessage(content="So you live in Noida, which is a major hub for the IT industry. It's likely that you have easy access to various tech companies and opportunities. Are you working for a company in Noida, or do you work remotely?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 51, 'prompt_tokens': 245, 'total_tokens': 296, 'completion_time': 0.119016207, 'completion_tokens_details': None, 'prompt_time': 0.015417192, 'prompt_tokens_details': None, 'queue_time': 0.15861303, 'total_time': 0.134433399}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_7ccc667439', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e64e3-ea5e-7e02-a3e6-20e96bce66b0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 245, 'output_tokens